# Chart Q2 2024's sessions and revenue

*Data Analysis in Python · Topic 8.1*

The weekly report needs two charts for April to June 2024: daily sessions as a line, and revenue by channel group as bars, with the biggest channel on top. Label both, so they can go in the report as they are.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell, the cell that builds the `sessions` table (lesson 3.'s steps) and the cell that adds its `channel_group` column (topic 4.2.'s steps).
2. In the first cell marked ✏️, create:
   - `daily_sessions`: the `sessions` table's sessions totaled by date, as a Series with the dates as its index
   - `trend_ax`: the Axes of a line chart of `daily_sessions`, with a title and a y-axis label
3. In the second cell marked ✏️, create:
   - `channel_revenue`: the `sessions` table's revenue totaled by `channel_group`, as a Series
   - `channel_ax`: the Axes of a horizontal bar chart of `channel_revenue`, with the biggest channel's bar on top, a title and an x-axis label
4. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: `daily_sessions` has 91 days, from 2,002 sessions on May 3 to 3,605 on June 25, 214,756 in all. `trend_ax` has one line of those 91 values, a title and a y-axis label. `channel_revenue` has nine channel groups adding up to $363,459.64. `channel_ax` has nine horizontal bars, Organic Search on top with $95,553.73 and Display at the bottom with $7,153.24, a title and an x-axis label.

## Parameters

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
MONTHS = ["2024-04", "2024-05", "2024-06"]
START, END = "2024-04-01", "2024-06-30"

In [ ]:
# The clean sessions table, with lesson 3's steps. Run it, and leave it as it is.
DAY_FIRST_MONTHS = ["2025-03"]
SOURCE_ALIASES = {
    "facebook.com": "facebook", "m.facebook.com": "facebook", "l.facebook.com": "facebook",
    "lm.facebook.com": "facebook", "fb": "facebook",
    "instagram.com": "instagram", "l.instagram.com": "instagram", "ig": "instagram",
}


def read_export(month):
    """Read one month's export with dates as dates, revenue as numbers and lost campaigns as missing."""
    export = pd.read_csv(DATA_URL + f"sessions_export_{month.replace('-', '_')}.csv",
                         na_values={"session_campaign": ["(not set)", "-"]})
    date_format = "%d/%m/%Y" if month in DAY_FIRST_MONTHS else "%Y-%m-%d"
    export["date"] = pd.to_datetime(export["date"], format=date_format)
    if not pd.api.types.is_numeric_dtype(export["revenue"]):
        export["revenue"] = export["revenue"].str.replace("$", "").str.replace(",", "").astype(float)
    return export


raw = pd.concat([read_export(month) for month in MONTHS], ignore_index=True)
raw["session_campaign"] = raw["session_campaign"].fillna("(not set)")
deduped = raw.drop_duplicates()
sessions = deduped[deduped["date"].between(START, END) & (deduped["session_medium"] != "test")].reset_index(drop=True)
sessions["source"] = sessions["session_source"].str.lower().str.strip().replace(SOURCE_ALIASES)
sessions.shape

In [ ]:
# Channel groups, with topic 4.2's steps and Foundations topic 12.3's function. Run it, and leave it as it is.
SOCIAL_SITES = {"facebook.com", "m.facebook.com", "l.facebook.com", "lm.facebook.com", "instagram.com", "l.instagram.com"}


def clean(value):
    """Return the value lowercased and stripped, or "" for None."""
    if value is None:
        return ""
    return value.strip().lower()


def channel_group(source, medium):
    """Return the channel group of a session's source and medium, with Foundations topic 12.3's rules."""
    source = clean(source)
    medium = clean(medium)
    if source == "(direct)" and medium == "(none)":
        return "Direct"
    if medium == "email":
        return "Email"
    if medium == "cpc":
        return "Paid Search"
    if medium == "display":
        return "Display"
    if medium == "paid_social":
        return "Paid Social"
    if medium == "video":
        return "Paid Video"
    if medium == "organic":
        return "Organic Search"
    if medium == "referral" and source in SOCIAL_SITES:
        return "Organic Social"
    if medium == "referral":
        return "Referral"
    return "Unassigned"


def row_channel_group(row):
    """Return the channel group of one row of the sessions table."""
    return channel_group(row["session_source"], row["session_medium"])


sessions["channel_group"] = sessions.apply(row_channel_group, axis=1)
sessions["channel_group"].value_counts()

## The trend

In [ ]:
# ✏️ Create daily_sessions and trend_ax


## The channels

In [ ]:
# ✏️ Create channel_revenue and channel_ax


In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables, your charts and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import matplotlib.axes
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    def bars(ax):
        """Return the bars' values from bottom to top, or left to right, and whether they're horizontal."""
        patches = [patch for patch in ax.patches if patch.get_width() and patch.get_height()]
        if not patches:
            return [], None
        horizontal = len({round(patch.get_height(), 6) for patch in patches}) == 1 and len({round(patch.get_width(), 6) for patch in patches}) > 1
        if horizontal:
            ordered = sorted(patches, key=lambda patch: patch.get_y())
            ordered = ordered[::-1] if ax.yaxis_inverted() else ordered
            return [patch.get_width() for patch in ordered], True
        ordered = sorted(patches, key=lambda patch: patch.get_x())
        ordered = ordered[::-1] if ax.xaxis_inverted() else ordered
        return [patch.get_height() for patch in ordered], False

    daily = found.get("daily_sessions")
    trend_ax = found.get("trend_ax")
    if isinstance(daily, pd.DataFrame) and daily.shape[1] == 1:
        daily = daily.iloc[:, 0]
    if not isinstance(daily, pd.Series):
        results.append((False, "daily_sessions should be a Series: the sessions table's sessions totaled by date, sessions.groupby(\"date\")[\"sessions\"].sum()."))
    elif len(daily) != 91 or daily.sum() != 214756:
        results.append((False, f"daily_sessions should have 91 days and 214,756 sessions, one value per day. It has {len(daily):,} values adding up to {daily.sum():,}."))
    elif not isinstance(trend_ax, matplotlib.axes.Axes):
        results.append((False, f"trend_ax should be the Axes of your line chart: fig, trend_ax = plt.subplots(), then trend_ax.plot(...), or trend_ax = daily_sessions.plot(). It is a {type(trend_ax).__name__}."))
    elif not trend_ax.get_lines():
        results.append((False, "trend_ax has no line on it. Draw daily_sessions on it: trend_ax.plot(daily_sessions.index, daily_sessions), or daily_sessions.plot(ax=trend_ax)."))
    elif any(len(line.get_ydata()) == 5149 for line in trend_ax.get_lines()):
        results.append((False, "trend_ax's line has 5,149 points: one per row of the sessions table, so it zigzags through every source and campaign of each day. Plot daily_sessions, one point per day."))
    elif len(trend_ax.get_lines()) != 1:
        results.append((False, f"trend_ax should have one line, daily sessions. It has {len(trend_ax.get_lines())}."))
    elif len(trend_ax.get_lines()[0].get_ydata()) != 91 or abs(sum(trend_ax.get_lines()[0].get_ydata()) - 214756) > 0.5:
        results.append((False, f"trend_ax's line should have 91 points, the daily sessions, adding up to 214,756. It has {len(trend_ax.get_lines()[0].get_ydata())} points adding up to {sum(trend_ax.get_lines()[0].get_ydata()):,.2f}."))
    elif not trend_ax.get_title().strip():
        results.append((False, "trend_ax has no title. Say what and when: trend_ax.set_title(\"Daily sessions, Q2 2024\")."))
    elif not trend_ax.get_ylabel().strip():
        results.append((False, "trend_ax has no y-axis label. Say what the numbers are: trend_ax.set_ylabel(\"Sessions\")."))
    else:
        results.append((True, f"The line chart has the 91 days' sessions, from {daily.min():,} on {daily.idxmin():%B %-d} to {daily.max():,} on {daily.idxmax():%B %-d}, and it's titled {trend_ax.get_title()!r}."))

    by_channel = found.get("channel_revenue")
    channel_ax = found.get("channel_ax")
    if isinstance(by_channel, pd.DataFrame) and by_channel.shape[1] == 1:
        by_channel = by_channel.iloc[:, 0]
    if not isinstance(by_channel, pd.Series):
        results.append((False, "channel_revenue should be a Series: the sessions table's revenue totaled by channel_group."))
    elif len(by_channel) != 9 or abs(by_channel.sum() - 363459.64) > 0.01:
        results.append((False, f"channel_revenue should have nine channel groups adding up to $363,459.64. It has {len(by_channel)}, adding up to {by_channel.sum():,.2f}."))
    elif not isinstance(channel_ax, matplotlib.axes.Axes):
        results.append((False, f"channel_ax should be the Axes of your bar chart: fig, channel_ax = plt.subplots(), then channel_revenue.plot.barh(ax=channel_ax). It is a {type(channel_ax).__name__}."))
    else:
        values, horizontal = bars(channel_ax)
        if len(values) != 9:
            results.append((False, f"channel_ax should have nine bars, one per channel group. It has {len(values)}."))
        elif horizontal is False:
            results.append((False, "channel_ax's bars are vertical. Make them horizontal, so the channel names fit: plot.barh() or ax.barh()."))
        elif abs(sum(values) - 214756) < 0.5:
            results.append((False, "channel_ax's bars add up to 214,756: those are sessions. Plot channel_revenue, the revenue."))
        elif abs(sum(values) - 363459.64) > 0.01:
            results.append((False, f"channel_ax's bars should be channel_revenue's values, adding up to $363,459.64. They add up to {sum(values):,.2f}."))
        elif values == sorted(values, reverse=True) and values != sorted(values):
            results.append((False, "channel_ax's biggest bar is at the bottom. matplotlib draws the first row at the bottom, so sort ascending: channel_revenue.sort_values().plot.barh()."))
        elif values != sorted(values):
            results.append((False, "channel_ax's bars aren't sorted by value: they're in the order of channel_revenue's index. Sort it first: channel_revenue.sort_values().plot.barh()."))
        elif not channel_ax.get_title().strip():
            results.append((False, "channel_ax has no title. Say what and when: channel_ax.set_title(\"Revenue by channel group, Q2 2024\")."))
        elif not channel_ax.get_xlabel().strip():
            results.append((False, "channel_ax has no x-axis label. Say what the bars measure: channel_ax.set_xlabel(\"Revenue ($)\")."))
        else:
            results.append((True, f"The bar chart has nine channels, sorted, with ${values[-1]:,.2f} on top and ${values[0]:,.2f} at the bottom, and it's titled {channel_ax.get_title()!r}."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
daily_sessions = sessions.groupby("date")["sessions"].sum()
fig, trend_ax = plt.subplots(figsize=(10, 4))
trend_ax.plot(daily_sessions.index, daily_sessions)
trend_ax.set_title("Daily sessions, Q2 2024")
trend_ax.set_ylabel("Sessions")
plt.show()
```

```python
channel_revenue = sessions.groupby("channel_group")["revenue"].sum()
fig, channel_ax = plt.subplots(figsize=(8, 4))
channel_revenue.sort_values().plot.barh(ax=channel_ax)
channel_ax.set_title("Revenue by channel group, Q2 2024")
channel_ax.set_xlabel("Revenue ($)")
channel_ax.set_ylabel("")
plt.show()
```

</details>